# Load groundwork 

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


import scanpy as sc

import os

import scvi

import seaborn as sns

%matplotlib inline


/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing CSCDataset from `anndata.experimental` is deprecated. Import anndata.abc.CSCDataset instead.
  warnings.warn(msg, FutureWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndat

In [12]:
## Downloaded CR-arc count matrix results
data_dir = '/mnt/hdd_bob/syy/adipose/atac/protocol_benchmark/downloaded_counts/VIB_10xmultiome_2'

h5_file = os.path.join(data_dir, 'GSM7102997_VIB_10xmultiome_2-outs-raw_feature_bc_matrix-matrix.h5ad')

barcodes_file = os.path.join(data_dir, 'GSM7102997_VIB_10xmultiome_2.barcodes.tsv')

features_file = os.path.join(data_dir, 'GSM7102997_VIB_10xmultiome_2.features.tsv')

h5 = sc.read_h5ad(h5_file)

features_pd = pd.read_csv(features_file, header=None, sep='\t')

barcodes_pd = pd.read_csv(barcodes_file, header=None, sep='\t')

## It's feature -by- barcode matrix, need to transpose
h5.var = barcodes_pd
h5.obs = features_pd 
h5.obs.columns = ['feature_id', 'name', 'feature_type', 'chrom', 'start', 'end']
h5.var.columns = ['barcodes']




In [13]:
h5

AnnData object with n_obs × n_vars = 104068 × 687495
    obs: 'feature_id', 'name', 'feature_type', 'chrom', 'start', 'end'
    var: 'barcodes'

## Matching atac bc with RNA bc

In [29]:
# atac-rna-barcode map 
bc_map_file = '/home/syyang/adipose_ln/multiom/atac_rna_barcodes_map.tsv'
bc_map_pd = pd.read_csv(bc_map_file, sep='\t')



In [30]:
bc_map_pd.head()

,atac_barcodes,rna_barcodes
0,ACAGCGGGTGTGTTAC,AAACAGCCAAACAACA
1,ACAGCGGGTTGTTCTT,AAACAGCCAAACATAG
2,ACAGCGGGTAACAGGC,AAACAGCCAAACCCTA
3,ACAGCGGGTGCGCGAA,AAACAGCCAAACCTAT
4,ACAGCGGGTCCTCCAT,AAACAGCCAAACCTTG


# RNA ad

In [36]:
h5_rna_ = h5.transpose()
h5_rna_.obs['rna_barcodes'] = h5_rna_.obs['barcodes'].map(lambda x: x.split('-1')[0])
h5_rna_.obs = h5_rna_.obs.merge(bc_map_pd, left_on='rna_barcodes', right_on='rna_barcodes', how='inner')
# h5_rna_.obs.set_index('rna_barcodes', inplace=True)

# select gene expression only, 
h5_rna_ = h5_rna_[:, h5_rna_.var['feature_type'] == 'Gene Expression']


# calculate total RNA QC metrics
h5_rna_.obs['total_RNA_UMI'] = h5_rna_.X.sum(axis=1)
h5_rna_.obs['total_RNA_UMI_log'] = np.log1p(h5_rna_.obs['total_RNA_UMI'])
h5_rna_.obs['n_genes_by_RNA_counts'] = (h5_rna_.X > 0).sum(axis=1)
h5_rna_.obs['N_MT_counts'] = h5_rna_.X[:, h5_rna_.var.name.apply(lambda x: x.startswith('MT-'))].sum(axis=1)
h5_rna_.obs['MT%'] = h5_rna_.obs['N_MT_counts'] / h5_rna_.obs['total_RNA_UMI'] * 100

# Add gene_name column 
h5_rna_.var['gene_name'] = h5_rna_.var['name'].astype(str)

/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/tmp/ipykernel_199051/2506221018.py:11: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  h5_rna_.obs['total_RNA_UMI'] = h5_rna_.X.sum(axis=1)
/home/syyang/python_virtuenv/mrvi_3.11/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [41]:
h5_rna_.var.head()

,feature_id,name,feature_type,chrom,start,end,gene_name
0,ENSG00000243485,MIR1302-2HG,Gene Expression,chr1,29553,30267,MIR1302-2HG
1,ENSG00000237613,FAM138A,Gene Expression,chr1,36080,36081,FAM138A
2,ENSG00000186092,OR4F5,Gene Expression,chr1,65418,69055,OR4F5
3,ENSG00000238009,AL627309.1,Gene Expression,chr1,120931,133723,AL627309.1
4,ENSG00000239945,AL627309.3,Gene Expression,chr1,91104,91105,AL627309.3


In [38]:
h5_rna_.shape

(687495, 36601)

In [44]:
entropy_dir = '/mnt/hdd_bob/syy/adipose/atac/res/VIB_10xmultiome_2_WS3000F'
RNA_info_dir = os.path.join(entropy_dir, 'RNA_info')
h5_rna_.write_h5ad(os.path.join(RNA_info_dir, 'allbc_rna.h5ad'))

In [52]:
union_bc_file = os.path.join(entropy_dir, '_cell_palling_comparison/Union_3set_atac_BCs.tsv')
union_bc_df = pd.read_csv(union_bc_file,  sep='\t')

union_bc_h5_rna_ = h5_rna_[h5_rna_.obs['atac_barcodes'].isin(union_bc_df['atac_bc'])].copy()

union_bc_h5_rna_.write_h5ad(os.path.join(entropy_dir, '_cell_palling_comparison/Union_3set_rna.h5ad'))